# Project 2.2: training runner

Runs the shared training loop from the repo on Colab (T4 GPU). Settings live in `configs/default.yaml`; feature and output formats are in `docs/data_contract.md`. Results are saved to `MyDrive/vizwiz_fusion/runs/<model>/seed<k>/`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
REPO = "/content/vizwiz-answerability-fusion"
if not os.path.exists(REPO):
    !git clone -q https://github.com/vishnuvardhan0ffl/vizwiz-answerability-fusion.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}

## One-off: convert the pilot cache

Creates `features/trainval` from the pilot notebook's cached features. It skips files that already exist, so rerunning is safe.

In [ ]:
!python scripts/convert_pilot_cache.py

## Quick check that the code works

In [ ]:
!python -m tests.test_smoke

## Train

Until the `test` part exists, train on `trainval` only. This gives dev scores and no test predictions.

In [ ]:
!python -m src.train --config configs/default.yaml --model all --seeds 0 --parts trainval

Once `features/test` exists, train every model with all three seeds:

In [ ]:
# !python -m src.train --config configs/default.yaml --model all

## Results so far

In [ ]:
import glob, json
import pandas as pd
rows = [json.load(open(p)) for p in glob.glob("/content/drive/MyDrive/vizwiz_fusion/runs/*/seed*/summary.json")]
cols = ["model", "seed", "best_dev_auprc", "epochs_run", "sec_per_epoch", "params", "peak_vram_gb"]
pd.DataFrame(rows).reindex(columns=cols).sort_values(["model", "seed"])